# ASR Inference Demo

This notebook demonstrates how to:

1. Clone and install the project.
2. Download the final model checkpoint and language model.
3. Run inference on a sample dataset.
4. Calculate WER and CER.
5. Run inference on a user-provided dataset.

A custom dataset must have the following structure:

```text
dataset/
├── audio/
│   ├── utterance_1.wav
│   ├── utterance_2.flac
│   └── utterance_3.mp3
└── transcriptions/          # optional
    ├── utterance_1.txt
    ├── utterance_2.txt
    └── utterance_3.txt

In [ ]:
REPO_URL = "https://github.com/antipula38/asr-homework.git"
REPO_DIR = "asr-homework"

!git clone $REPO_URL
%cd $REPO_DIR

## Installation

Install all packages listed in `requirements.txt`.

In [ ]:
!python -m pip install --upgrade pip
!pip install -r requirements.txt

## Download model resources

Download the final checkpoint and the external language model required for
Beam Search + LM decoding.

In [ ]:
!python download_resources.py

In [ ]:
from pathlib import Path

checkpoint = Path("saved/final/model_best.pth")
language_model = Path("resources/lm/3-gram.pruned.1e-7.arpa")

print("Checkpoint:", checkpoint.exists())
print("Language model:", language_model.exists())

data/demo/
├── audio/
│   ├── example_1.wav
│   └── example_2.wav
└── transcriptions/
    ├── example_1.txt
    └── example_2.txt

In [ ]:
## Sample dataset

The repository contains a small example dataset in the same format accepted
by `CustomDirDataset`.

In [ ]:
from pathlib import Path

demo_dir = Path("data/demo")

print("Audio files:")
for path in sorted((demo_dir / "audio").glob("*")):
    print(" ", path.name)

print("\nReference transcriptions:")
for path in sorted((demo_dir / "transcriptions").glob("*.txt")):
    print(" ", path.name, "->", path.read_text(encoding="utf-8").strip())

## Run inference

Run the final EMA model with Beam Search + language model decoding.
Predictions are written to `.txt` files with the same IDs as the input audio.

In [ ]:
!rm -rf predictions/demo

!python inference.py \
    inferencer.from_pretrained="saved/final/model_best.pth" \
    datasets.test.data_dir="data/demo" \
    inferencer.save_path="predictions/demo"

In [ ]:
prediction_dir = Path("predictions/demo/test")

for path in sorted(prediction_dir.glob("*.txt")):
    print(path.name, "->", path.read_text(encoding="utf-8").strip())

In [ ]:
!python calc_metrics.py \
    --ground-truth="data/demo/transcriptions" \
    --predictions="predictions/demo/test"

## Run inference on your own dataset

Set `DATASET_URL` to a link to a `.zip` archive containing your dataset.

Google Drive links and regular direct download links are supported.

The archive must contain either:

```text
audio/
transcriptions/

In [ ]:
DATASET_URL = ""

# Example:
# DATASET_URL = "https://drive.google.com/file/d/FILE_ID/view?usp=sharing"

assert DATASET_URL, "Please set DATASET_URL before running this cell."

## Inference on the custom dataset

The following cell runs the same `inference.py` script on the downloaded
dataset.

If reference transcriptions are available, `calc_metrics.py` is also run
automatically.

In [ ]:
import subprocess
import shutil
from pathlib import Path


output_dir = Path("predictions/user_dataset")
shutil.rmtree(output_dir, ignore_errors=True)


inference_command = [
    "python",
    "inference.py",
    'inferencer.from_pretrained=saved/final/model_best.pth',
    f'datasets.test.data_dir={user_dataset_dir}',
    'inferencer.save_path=predictions/user_dataset',
]

subprocess.run(inference_command, check=True)


prediction_dir = output_dir / "test"

print("\nPredictions:")

for path in sorted(prediction_dir.glob("*.txt")):
    print(path.name, "->", path.read_text(encoding="utf-8").strip())


ground_truth_dir = user_dataset_dir / "transcriptions"

if ground_truth_dir.is_dir():
    print("\nCalculating WER and CER...")

    subprocess.run(
        [
            "python",
            "calc_metrics.py",
            f"--ground-truth={ground_truth_dir}",
            f"--predictions={prediction_dir}",
        ],
        check=True,
    )
else:
    print(
        "\nNo transcriptions/ directory was provided. "
        "Inference is complete, but WER/CER cannot be calculated."
    )